# Driver Drowsiness Ground Truth Generation
 

This notebook prepares a driver drowsiness  ground truth dataset from raw in-cabin video and then evaluates a Vision-Language Model (VLM) on the same clips. The core idea is to work with short temporal snippets (8 frames over 2 seconds) instead of individual frames, so that temporal patterns such as prolonged eye closure and head nodding are preserved while keeping computation and annotation effort manageable [web:176][web:166].  

The workflow consists of:  
- Extracting full-frame sequences from raw video for sanity checking  
- Deriving non-overlapping 2-second windows with 8 evenly spaced frames  
- Creating human annotation templates with physiologically grounded rules  
- Measuring inter-annotator agreement with Cohen’s Kappa  
- Comparing Qwen2.5-VL predictions against adjudicated human ground truth [web:176][web:162].  


In [2]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


2. Import Libraries

These libraries handle video I/O, file paths, and directory creation.

In [3]:
import cv2
import json
import numpy as np
import os
from pathlib import Path

3. Set Input & Output Paths

Modify input.mp4 to the name of your video file.

In [4]:
input_path = r"C:\Users\nikhi\Downloads\5.mp4\5.mp4" # replace with your video
out_dir = Path('frames1')
out_dir.mkdir(parents=True, exist_ok=True)

4. Read Video Information

We gather FPS and duration to determine extraction behavior.

In [5]:
cap = cv2.VideoCapture(str(input_path))

if not cap.isOpened():
    raise SystemExit(f"Cannot open {input_path}")

video_fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
duration = frame_count / video_fps if video_fps else 0

print(f"Video FPS (detected): {video_fps}")
print(f"Total Frames: {frame_count}, Duration: {duration:.2f} seconds")


Video FPS (detected): 30.0
Total Frames: 18224, Duration: 607.47 seconds


5. Extract Frames at 30 FPS

In [6]:
target_fps = 30.0

frame_index = 0
written = 0

if video_fps and abs(video_fps - target_fps) < 1e-6:
    # Save every frame
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        out_name = out_dir / f"frame_{frame_index:06d}.png"
        cv2.imwrite(str(out_name), frame)
        frame_index += 1
        written += 1

else:
    # Sample by timestamp
    t = 0.0
    max_t = duration if duration else 1e12

    while t <= max_t:
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000.0)
        ret, frame = cap.read()
        if not ret:
            break
        out_name = out_dir / f"frame_{written:06d}_t{t:.3f}s.png"
        cv2.imwrite(str(out_name), frame)
        written += 1
        t += 1.0 / target_fps

cap.release()
print(f"Done — extracted {written} frames into '{out_dir}'")


Done — extracted 18224 frames into 'frames1'


Before running the clip extraction function, the notebook specifies all input and output locations using Python’s `pathlib.Path` for platform-independent path handling [web:233]. The following configuration ensures all generated clips and annotation files are kept organized and reproducible:

- `VIDEO_PATH`: Full path to the input driving video.  
- `ROOT_DIR`: Main output directory for all results and intermediate files.  
- `CLIP_OUT_DIR`: Target subdirectory for extracted 8-frame clips, created if it does not already exist.  
- `video_label`: Metadata dictionary (e.g., for legacy drowsiness labels), defaulting to `"unknown"` for new raw videos.

This structure makes it easy to process multiple videos and keep 8-frame annotated clips organized for downstream manual annotation and VLM benchmarking.

In [7]:
VIDEO_PATH = Path(r"C:\Users\nikhi\Downloads\5.mp4\5.mp4")  # same video
ROOT_DIR = Path(r"C:\Users\nikhi\Downloads\drowsiness_gt")
CLIP_OUT_DIR = ROOT_DIR / "clips_8f"
CLIP_OUT_DIR.mkdir(parents=True, exist_ok=True)

video_label = {"drowsiness_label": "unknown"}

## Extract 8-Frame, 2-Second Clips

The function `extract_8frame_clips_from_video` converts the continuous driving video into non-overlapping 2‑second clips, each represented by 8 evenly spaced frames. This design is motivated by action recognition work showing that very short snippets of about 5–7 frames already capture most of the discriminative temporal information for human actions, with diminishing returns beyond that point [Schindler & Van Gool, 2008](https://ieeexplore.ieee.org/document/4587730)[web:176]. Using 8 frames provides a small safety margin while keeping each snippet compact for efficient annotation and model inference.  

For each input video, the function:
- Opens the file with `cv2.VideoCapture`, reads its FPS and total frame count, and computes the duration, following standard OpenCV video I/O usage [OpenCV Video I/O Tutorial](https://docs.opencv.org/4.x/dd/d43/tutorial_py_video_display.html).  
- Divides the timeline into consecutive 2‑second windows and, in each window, uses `numpy.linspace` to select 8 frame indices spread uniformly across the segment.  
- Seeks to each selected frame, saves it as `frame_XX_fYYYY.jpg` inside a per‑clip folder, and records metadata such as frame indices and timestamps in a `clips_index` list.  

This yields a set of temporally coherent 8‑frame snippets at an effective sampling rate of roughly 4 Hz (for ≈30 FPS video), which is sufficient to resolve normal blinks versus prolonged closures and microsleeps on the 0.4–1.0+ second timescale that drowsiness metrics like PERCLOS are based on, as reported in blink and drowsiness literature [Caffier et al., 2003](https://pubmed.ncbi.nlm.nih.gov/12736840/)[Dinges & Grace, 1998](https://rosap.ntl.bts.gov/view/dot/113).  


In [8]:
def extract_8frame_clips_from_video(
    video_path: Path,
    video_label: dict,
    clip_out_dir: Path,
    clip_duration_sec: float = 2.0,
    num_frames_per_clip: int = 8,
):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"❌ Cannot open {video_path}")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    duration = frame_count / fps if fps else 0.0

    print(f"📹 {video_path.name}")
    print(f"   FPS: {fps:.2f} | Frames: {frame_count:,} | Duration: {duration:.1f}s")

    clip_length_frames = int(round(clip_duration_sec * fps))
    if clip_length_frames < num_frames_per_clip:
        print(f"   ⚠ Skipping: video too short for {clip_duration_sec}s clip")
        cap.release()
        return []

    clips_index = []
    clip_id_counter = 0

    for start_frame in range(0, frame_count - clip_length_frames + 1, clip_length_frames):
        frame_indices = np.linspace(
            start_frame,
            start_frame + clip_length_frames - 1,
            num_frames_per_clip,
            dtype=int
        ).tolist()

        clip_id = f"{video_path.stem}_clip_{clip_id_counter:04d}"
        clip_dir = clip_out_dir / clip_id
        clip_dir.mkdir(parents=True, exist_ok=True)

        saved_frames = []
        for i, frame_idx in enumerate(frame_indices):
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if not ret:
                break

            frame_name = f"frame_{i:02d}_f{frame_idx}.jpg"
            out_path = clip_dir / frame_name
            cv2.imwrite(str(out_path), frame)
            saved_frames.append({
                "frame_file": frame_name,
                "frame_idx": int(frame_idx),
                "timestamp_sec": frame_idx / fps
            })

        if len(saved_frames) == num_frames_per_clip:
            clip_meta = {
                "clip_id": clip_id,
                "video_file": video_path.name,
                "frame_indices": [f["frame_idx"] for f in saved_frames],
                "timestamps_sec": [f["timestamp_sec"] for f in saved_frames],
                "dataset_drowsiness_label": video_label.get("drowsiness_label", "unknown"),
                "original_metadata": video_label
            }
            clips_index.append(clip_meta)

        clip_id_counter += 1

    cap.release()
    print(f"   ✓ Extracted {len(clips_index)} clips")
    return clips_index


After defining the 8‑frame extraction function, this step applies it to the configured input video and writes a global JSON index of all generated clips. The call to `extract_8frame_clips_from_video` processes the entire video into non-overlapping 2‑second windows, each represented by 8 evenly spaced frames, following the short action-snippet rationale from Schindler and Van Gool’s work on minimal temporal context for action recognition [Schindler & Van Gool, 2008](https://ieeexplore.ieee.org/document/4587730)[web:176].  

The resulting `clips_index` list is then saved as `clips_index.json` inside the `clips_8f` output directory using Python’s built-in `json` module, which provides a reproducible mapping from each `clip_id` to its source video file, frame indices, and timestamps [Python json module docs](https://docs.python.org/3/library/json.html)[web:258]. This index serves as the central reference for subsequent steps in the pipeline, including human annotation, inter-annotator agreement computation, and VLM evaluation.

In [9]:
clips_index = extract_8frame_clips_from_video(
    VIDEO_PATH,
    video_label,
    CLIP_OUT_DIR,
    clip_duration_sec=2.0,
    num_frames_per_clip=8
)

index_path = CLIP_OUT_DIR / "clips_index.json"
with open(index_path, "w") as f:
    json.dump(clips_index, f, indent=2)

print(f"\n✓ Total clips extracted: {len(clips_index)}")
print(f"✓ Clips index saved: {index_path}")


📹 5.mp4
   FPS: 30.00 | Frames: 18,224 | Duration: 607.5s
   ✓ Extracted 303 clips

✓ Total clips extracted: 303
✓ Clips index saved: C:\Users\nikhi\Downloads\drowsiness_gt\clips_8f\clips_index.json
   ✓ Extracted 303 clips

✓ Total clips extracted: 303
✓ Clips index saved: C:\Users\nikhi\Downloads\drowsiness_gt\clips_8f\clips_index.json


## Overview: End-to-End Qwen3-VL Drowsiness Evaluation via Hugging Face

This section uses **Qwen3-VL** (a multimodal vision-language model from Hugging Face) to automatically classify driver drowsiness from 8-frame video clips.

### What we are Building
- Input: 120 selected 8-frame clips (2 seconds each) from driver video
- Process: Send all 8 frames to Qwen in a single request (preserving temporal context)
- Output: Structured JSON predictions with drowsiness indicators (occlusion, lighting, posture, eye visibility, etc.)
- Use case: Compare VLM predictions against human ground truth annotations to evaluate model performance

### Key Design Principles
1. **Temporal Coherence**: 8 frames (not single frames) preserve motion patterns crucial for drowsiness detection (head nods, eye blinks, microsleeps)
2. **Single Scene Analysis**: All 8 frames processed together as ONE unified scene, not separate batches
3. **Evidence-Based Criteria**: Prompt grounds Qwen in PERCLOS and blink literature [Dinges & Grace, 1998; Caffier et al., 2003]
4. **JSON-Only Output**: Strict format ensures reliable parsing and downstream comparison with ground truth



In [1]:
import os
import json
import time
import base64
from pathlib import Path
from typing import List
import requests
from PIL import Image

In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5vl:3b-q8_0"

In [4]:


CLIPS_DIR = Path(r"C:\Users\nikhi\Downloads\drowsiness_gt\clips_8f")
OUTPUT_DIR = Path(r"C:\Users\nikhi\Downloads\drowsiness_gt\qwen_output_8f_120_final_v2")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)


In [5]:
# ----------------------------
# SYSTEM PROMPT (strict JSON schema with occlusion & notes)
# ----------------------------
SYSTEM_PROMPT = """
SYSTEM: You are Qwen2.5-VL (qwen2.5vl:3b-q8_0) for automated driver behavior evaluation.
You will receive 8 consecutive frames from a driver's face/upper torso.
Produce JSON-only output following exactly this schema:

{
  "clip_id": "<string>",
  "drowsy": {"label": "yes"|"no","confidence": 0.0},
  "behaviors": {
      "yawn": {"detected": false,"confidence":0.0},
      "cover_mouth": {"detected": false,"confidence":0.0},
      "eat": {"detected": false,"confidence":0.0},
      "drink": {"detected": false,"confidence":0.0},
      "sneeze": {"detected": false,"confidence":0.0},
      "cry": {"detected": false,"confidence":0.0},
      "micro_sleep": {"detected": false,"confidence":0.0},
      "talking": {"detected": false,"confidence":0.0},
      "hands_on_face": {"detected": false,"confidence":0.0},
      "glasses": {"detected": false,"confidence":0.0},
      "sunglasses": {"detected": false,"confidence":0.0},
      "mask": {"detected": false,"confidence":0.0},
      "other_occlusion": {"detected": false,"confidence":0.0}
  },
  "eye_visibility": {"left_eye":"open","right_eye":"open"},
  "mouth_visibility": "open"|"closed"|"occluded",
  "head_posture": {"pitch":"neutral","yaw":"center","roll":"neutral"},
  "occlusion": {"face_occluded": true|false, "reason":"mask"|"hand"|"sun_glare"|"other"|null},
  "lighting": {"level":"normal","issue":null},
  "evidence_frames": [0,1,2,3,4,5,6,7],
  "notes": ""  # give a short description summarizing the scene
}

- Always fill all fields, even if uncertain.
- Use numeric confidence scores 0.0-1.0.
- Do not output per-frame arrays, PERCLOS, blink rates, or markdown. Return only the JSON object.
"""

# ----------------------------
# USER PROMPT TEMPLATE (aggregated 2-second scene with occlusion and notes)
# ----------------------------
USER_PROMPT_TEMPLATE = """
You will analyze the following 8 consecutive frames from a driver's cabin using Qwen2.5VL:3b-q8_0.
Treat all 8 frames together as a single 2-second scene. Do NOT evaluate frames independently or produce per-frame outputs.
Aggregate all observed behaviors over the scene: yawning, covering mouth, eating, drinking, sneezing, crying, micro-sleep, eyes closed, looking away, talking, hands on face, wearing glasses/sunglasses, mask occlusion, or any other gestures.
Also report mouth visibility as "open", "closed", or "occluded".
Report face occlusion in "occlusion" field with reason (mask, hand, sun_glare, other, or null).
Provide a short textual summary of the scene in the "notes" field.

Clip id: {clip_id}
Frames: images[0..7] in order (0 = earliest, 7 = latest)

Answer only JSON that strictly conforms to the SYSTEM_PROMPT schema.
Always fill all fields, even if uncertain. Do NOT include per-frame outputs, metrics like PERCLOS or blink rates, or markdown.
"""


In [6]:
def load_and_encode_images(clip_folder: Path, target_long_side: int = 800) -> List[str]:
    images_b64 = []
    files = sorted([p for p in clip_folder.iterdir() if p.is_file()])[:8]
    if len(files) < 8:
        raise ValueError(f"Clip {clip_folder} has fewer than 8 images.")
    for p in files:
        img = Image.open(p).convert("RGB")
        w, h = img.size
        max_side = max(w, h)
        if max_side > target_long_side:
            scale = target_long_side / max_side
            img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
        with open(p, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode("ascii")
        images_b64.append(b64)
    return images_b64

In [7]:
def call_ollama(images_b64: List[str], clip_id: str):
    user_prompt = USER_PROMPT_TEMPLATE.format(clip_id=clip_id)
    data = {
        "model": MODEL_NAME,
        "system": SYSTEM_PROMPT,
        "prompt": user_prompt,
        "images": images_b64,
        "format": "json",
        "stream": False
    }
    resp = requests.post(OLLAMA_URL, json=data)
    resp.raise_for_status()
    return resp.json()

In [8]:
def process_clips(limit: int = 120):
    clip_folders = sorted([p for p in CLIPS_DIR.iterdir() if p.is_dir()])[:limit]
    for i, clip in enumerate(clip_folders, 1):
        clip_id = clip.name
        try:
            imgs_b64 = load_and_encode_images(clip)
            raw = call_ollama(imgs_b64, clip_id)

            # Extract JSON from 'response' field
            response_str = raw.get("response", "{}")
            try:
                parsed_json = json.loads(response_str)
            except json.JSONDecodeError:
                parsed_json = {"_raw_text": response_str}

            # Save only parsed JSON
            timestamp = int(time.time())
            out_file = OUTPUT_DIR / f"{clip_id}_parsed_{timestamp}.json"
            with open(out_file, 'w') as f:
                json.dump(parsed_json, f, indent=2)

            print(f"[{i}/{len(clip_folders)}] Saved output for {clip_id}")

        except Exception as e:
            print(f"Error processing {clip_id}: {e}")

In [9]:
if __name__ == "__main__":
    process_clips()

[1/120] Saved output for 5_clip_0000
[2/120] Saved output for 5_clip_0001
[3/120] Saved output for 5_clip_0002
[4/120] Saved output for 5_clip_0003
[5/120] Saved output for 5_clip_0004
[6/120] Saved output for 5_clip_0005
[7/120] Saved output for 5_clip_0006
[8/120] Saved output for 5_clip_0007
[9/120] Saved output for 5_clip_0008
[10/120] Saved output for 5_clip_0009
[11/120] Saved output for 5_clip_0010
[12/120] Saved output for 5_clip_0011
[13/120] Saved output for 5_clip_0012
[14/120] Saved output for 5_clip_0013
[15/120] Saved output for 5_clip_0014
[16/120] Saved output for 5_clip_0015
[17/120] Saved output for 5_clip_0016
[18/120] Saved output for 5_clip_0017
[19/120] Saved output for 5_clip_0018
[20/120] Saved output for 5_clip_0019
[21/120] Saved output for 5_clip_0020
[22/120] Saved output for 5_clip_0021
[23/120] Saved output for 5_clip_0022
[24/120] Saved output for 5_clip_0023
[25/120] Saved output for 5_clip_0024
[26/120] Saved output for 5_clip_0025
[27/120] Saved output